# Notebook 1: Setup & Connecting to Neo4j Aura

In this workshop, you will learn how to build **Knowledge Graphs** from different data sources using the `neo4j-graphrag` Python package. By the end, you'll be able to extract structured knowledge from:

- Plain text
- PDF documents
- Web pages

...and store it all in a Neo4j Aura graph database.

## What is a Knowledge Graph?

A knowledge graph is a structured representation of information where:
- **Nodes** represent entities (people, places, concepts)
- **Relationships** connect entities ("works at", "located in", "authored by")
- **Properties** store attributes on nodes and relationships

## What is `neo4j-graphrag`?

The `neo4j-graphrag` package provides tools to:
1. **Build** knowledge graphs from unstructured text using LLMs
2. **Query** knowledge graphs using retrieval-augmented generation (RAG)

We'll focus on building knowledge graphs in this workshop.

## Step 1: Install Dependencies

Run the cell below to install everything you need. If you've already activated the `.venv`, these should already be installed.

In [1]:
%pip install "neo4j-graphrag[openai]" python-dotenv aiohttp beautifulsoup4 -q

Note: you may need to restart the kernel to use updated packages.


## Step 2: Load Environment Variables

We store our credentials in a `.env` file so they don't end up in our code. The `.env` file should contain:

```
NEO4J_URI=neo4j+s://your-instance.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-password
NEO4J_DATABASE=neo4j
OPENAI_API_KEY=sk-your-key
```

In [2]:
import os
from dotenv import load_dotenv

# Load variables from ../.env (one directory up from notebooks/)
load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# Quick sanity check — make sure the variables loaded
assert NEO4J_URI, "NEO4J_URI not found in .env!"
assert NEO4J_PASSWORD, "NEO4J_PASSWORD not found in .env!"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in .env!"

print(f"Neo4j URI: {NEO4J_URI}")
print(f"Database:  {NEO4J_DATABASE}")
print("All credentials loaded successfully!")

Neo4j URI: neo4j+s://fc1f6bc7.databases.neo4j.io
Database:  fc1f6bc7
All credentials loaded successfully!


## Step 3: Connect to Neo4j Aura

We'll create a Neo4j driver — this is our connection to the database. The `neo4j+s://` URI scheme means we're connecting over an encrypted connection (required for Aura).

> **Note:** If your Aura instance has been paused (free tier instances auto-pause after inactivity), go to [console.neo4j.io](https://console.neo4j.io) and resume it first.

In [3]:
import neo4j

driver = neo4j.GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

# Test the connection
driver.verify_connectivity()
print("Connected to Neo4j Aura successfully!")

Connected to Neo4j Aura successfully!


## Step 4: Explore the Database

Let's see what's currently in the database. If this is a fresh instance, it should be empty.

In [4]:
# Count all nodes
records, _, _ = driver.execute_query(
    "MATCH (n) RETURN count(n) AS node_count",
    database_=NEO4J_DATABASE,
)
print(f"Total nodes in database: {records[0]['node_count']}")

# Count all relationships
records, _, _ = driver.execute_query(
    "MATCH ()-[r]->() RETURN count(r) AS rel_count",
    database_=NEO4J_DATABASE,
)
print(f"Total relationships in database: {records[0]['rel_count']}")

# Show node labels (types of entities)
records, _, _ = driver.execute_query(
    "CALL db.labels() YIELD label RETURN collect(label) AS labels",
    database_=NEO4J_DATABASE,
)
print(f"Node labels: {records[0]['labels']}")

Total nodes in database: 24
Total relationships in database: 42
Node labels: ['Movie', 'Genre', 'User', 'Person', 'Question', 'Answer', 'Comment', 'Tag', '__KGBuilder__', 'Document', 'Chunk', 'AIModel', '__Entity__', 'Organization', 'Benchmark']


## Step 5: Test OpenAI Connection

The `neo4j-graphrag` package uses an LLM to extract entities and relationships from text. Let's verify the API key works.

The `response_format` parameter tells the model to always return valid JSON — this is recommended by the documentation for the entity extractor.

In [ ]:
from neo4j_graphrag.llm import OpenAILLM

llm = OpenAILLM(
    model_name="gpt-4.1-2025-04-14",
    model_params={
        "temperature": 0,
        "response_format": {"type": "json_object"},
    },
)

# Quick test
response = llm.invoke("Return a JSON object with key 'status' and value 'ok'.")
print(f"LLM response: {response.content}")
print("OpenAI connection works!")

## Utility: Clear the Database

You can run this cell at any time to wipe the database and start fresh. **Use with caution!**

In [ ]:
# UNCOMMENT THE LINES BELOW TO CLEAR THE DATABASE
# This deletes ALL nodes and relationships!

# driver.execute_query("MATCH (n) DETACH DELETE n", database_=NEO4J_DATABASE)
# print("Database cleared!")

## Summary

You now have:
- A working connection to Neo4j Aura
- A working OpenAI API key
- All dependencies installed

**Next up:** [Notebook 2 — Building a Knowledge Graph from Plain Text](./02_kg_from_text.ipynb)

In [6]:
# Clean up the driver (we'll create a new one in each notebook)
driver.close()